# A Practical Guide to Quantitative Finance Interviews — Chapter 5
## Stochastic Process and Stochastic Calculus

Worked solutions to the stochastic-process problems in **Xinfeng Zhou's Green Book**. Each topic gets a
plain-language explanation with a small example, and every problem carries an explicit **transition graph**,
the **first-step (one-step) equations**, and a **solve-for-`n`** function with a Monte-Carlo check. Only the
Python standard library is used (`fractions` keeps every answer exact).

| § | Topic | Problems |
|---|-------|----------|
| **5.1** | Markov Chains | Gambler's ruin · Dice question · Coin triplets (A/B/C) · Color balls |
| **5.2** | Martingales & Random Walks | Drunk man · Dice game (Wald) · Ticket line (reflection) · Coin sequence (induction + martingale) |
| **5.3** | Dynamic Programming | Dice game · World series · Dynamic dice game · Dynamic card game · American & European options |
| **5.4** | Brownian Motion & Stochastic Calculus | Brownian motion · corr(B, B²) · P(B₁>0, B₂<0) · stopping time to ±1 · first-passage density · hitting with/without drift · never-reach · Itô's lemma · is W³ a martingale? |

## 5.1 Markov Chains

A **Markov chain** is a random process that hops between a set of **states** $\{1,2,\dots\}$ where the next
state depends **only on the current one**, not on the full history:
$$P(X_{t+1}=j\mid X_t=i,\ X_{t-1},\dots,X_0)=P(X_{t+1}=j\mid X_t=i)=P_{ij}.$$
This "memoryless" property (the **Markov property**) is what makes the chain tractable — the present state is a
complete summary of the past.

### The transition matrix
Collect the one-step probabilities into a matrix $P$ whose entry $P_{ij}$ is the chance of moving from state
$i$ to state $j$. Every **row is a probability distribution**, so each row sums to $1$:
$$\sum_{j} P_{ij}=1\quad\text{for every }i.$$

*Simple example (weather).* States $\{S=\text{sunny},\ R=\text{rainy}\}$. Say a sunny day is followed by sun
$80\%$ of the time and a rainy day by rain $60\%$:
$$P=\begin{pmatrix}P_{SS}&P_{SR}\\P_{RS}&P_{RR}\end{pmatrix}=\begin{pmatrix}0.8&0.2\\0.4&0.6\end{pmatrix}.$$

### The probability of a path
Because each step only looks at the current state, the probability of a whole **path** is the initial
probability times the product of the one-step probabilities along it:
$$P(X_0=i_0,X_1=i_1,\dots,X_n=i_n)=P(X_0=i_0)\,P_{i_0 i_1}P_{i_1 i_2}\cdots P_{i_{n-1}i_n}.$$
*Example.* Starting sunny, the path $S\to S\to R$ has probability $P_{SS}\,P_{SR}=0.8\times0.2=0.16$.

To jump several steps at once, **multiply the matrix**: the $n$-step probabilities are the entries of $P^{n}$,
$$P(X_{t+n}=j\mid X_t=i)=\big(P^{n}\big)_{ij}.$$
For the weather chain, $P^{2}=\begin{pmatrix}0.72&0.28\\0.56&0.44\end{pmatrix}$, so two sunny days from now is
$72\%$ likely if today is sunny.

### The transition graph
The same information draws as a **directed graph**: one node per state, and an arrow $i\to j$ labeled with
$P_{ij}$ for every non-zero transition (a self-loop $i\to i$ when $P_{ii}>0$). The weather chain:

```
          0.2  (S -> R)
       ┌──────────────►┐
  0.8 ⟲│  (S)     (R)  │⟲ 0.6
       └◄──────────────┘
          0.4  (R -> S)
```

Reading a graph and reading the matrix are interchangeable; the graph makes the *structure* (which states can
reach which) jump out, which is exactly what the next idea needs.

### Classification of states
- **Accessible / communicating.** $j$ is *accessible* from $i$ if some path of positive probability leads
  $i\to\cdots\to j$. If they are accessible from each other, they **communicate**. A chain in which every state
  communicates with every other is **irreducible**.
- **Recurrent vs. transient.** From a **recurrent** state the chain is *certain* to return eventually; from a
  **transient** state there is positive probability it never comes back. In a finite chain, once it leaves a
  transient state enough times it never returns.
- **Absorbing.** A state $i$ with $P_{ii}=1$ is **absorbing** — once entered, the chain stays forever. (In the
  weather example neither state is absorbing.)
- **Periodic vs. aperiodic.** A state has **period** $d$ if returns are only possible at multiples of $d$
  steps; $d=1$ is **aperiodic**. A self-loop makes a state aperiodic.

The problems below all reduce to the most useful special case in interviews: chains with absorbing states.

### Absorbing Markov chains
A chain is **absorbing** if (1) it has at least one absorbing state and (2) every state can reach one. Order
the states with the **transient** ones first and the **absorbing** ones last; the matrix then splits into the
**canonical form**
$$P=\begin{pmatrix}Q & R\\[2pt] \mathbf 0 & I\end{pmatrix},$$
where $Q$ (transient$\to$transient) and $R$ (transient$\to$absorbing) hold everything that can still change.
The eventual fate — *which* absorbing state, and *how long* until it is reached — is what we solve for.

### Equations for absorption probability (first-step analysis)
Let $a_i=P(\text{absorbed in a chosen target set}\mid X_0=i)$. Condition on the **first step** and use the
Markov property: from $i$ you land in $j$ with probability $P_{ij}$, then face the *same* question from $j$.
$$\boxed{\,a_i=\sum_{j}P_{ij}\,a_j\,}\qquad\text{with } a_i=1 \text{ on target absorbing states, } a_i=0 \text{ on the others.}$$
In matrix form (only the transient unknowns), $a=Qa+R\mathbf 1_{\text{target}}$, i.e. $(I-Q)\,a=R\mathbf 1_{\text{target}}$.

### Equations for expected time to absorption
Let $t_i=E[\text{steps until absorption}\mid X_0=i]$. The first step *costs 1*, then you continue from wherever
you land:
$$\boxed{\,t_i=1+\sum_{j\ \text{transient}}P_{ij}\,t_j\,}\qquad t_i=0 \text{ on absorbing states.}$$
In matrix form $(I-Q)\,t=\mathbf 1$. The inverse $N=(I-Q)^{-1}$ is the **fundamental matrix**: $N_{ij}$ is the
expected number of visits to $j$ starting from $i$, so $t=N\mathbf 1$ (row sums) and the absorption
probabilities are $B=NR$. The code cell below solves both systems exactly with rational arithmetic, and every
problem in this section is an instance of these two boxed equations.

In [1]:
from fractions import Fraction

def solve_linear(A, b):
    """Solve the linear system A x = b exactly over the rationals (Gaussian elimination with Fractions)."""
    n = len(A)
    M = [[Fraction(A[i][j]) for j in range(n)] + [Fraction(b[i])] for i in range(n)]
    for col in range(n):
        piv = next(r for r in range(col, n) if M[r][col] != 0)     # nonzero pivot
        M[col], M[piv] = M[piv], M[col]
        pv = M[col][col]
        M[col] = [v / pv for v in M[col]]
        for r in range(n):
            if r != col and M[r][col] != 0:
                f = M[r][col]
                M[r] = [M[r][k] - f * M[col][k] for k in range(n + 1)]
    return [M[i][n] for i in range(n)]

def absorbing_analysis(P, states, targets):
    """P: dict {state: {next_state: prob}} with rows summing to 1. An absorbing state has P[s]=={s:1}.
    Returns (absorb_prob, expected_time), each a dict over the transient states:
      absorb_prob[i] = P(absorbed in `targets` | start i)     solves (I-Q) a = R*1_target
      expected_time[i] = E[steps to absorption | start i]     solves (I-Q) t = 1
    -- the two boxed first-step equations from the notes above."""
    absorbing = [s for s in states if P[s].get(s, 0) == 1 and len(P[s]) == 1]
    transient = [s for s in states if s not in absorbing]
    m = len(transient)
    ImQ = [[Fraction(i == j) - Fraction(P[transient[i]].get(transient[j], 0)) for j in range(m)]
           for i in range(m)]
    t = solve_linear(ImQ, [Fraction(1)] * m)
    r = [sum(Fraction(P[transient[i]].get(s, 0)) for s in targets) for i in range(m)]
    a = solve_linear(ImQ, r)
    return {transient[i]: a[i] for i in range(m)}, {transient[i]: t[i] for i in range(m)}

# quick self-test on the 2-state weather chain (no absorbing states -> just checks the linear solver)
demo = solve_linear([[Fraction(1), Fraction(-1)], [Fraction(2), Fraction(1)]], [Fraction(0), Fraction(3)])
print("linear-solver sanity check x =", demo, "(expected [1, 1])")

linear-solver sanity check x = [Fraction(1, 1), Fraction(1, 1)] (expected [1, 1])


### 5.1.1 Gambler's ruin problem

**Problem.** Player $M$ starts with 1 dollar and player $N$ with 2 dollars; each game the winner takes a
dollar from the loser. $M$ (the better player) wins each game with probability $\tfrac23$. They play until
someone is broke. What is $P(M\text{ wins})$?

**State space.** The two stakes always sum to 3 dollars, so a single number — $M$'s money
$m\in\{0,1,2,3\}$ — describes everything. States $0$ ($M$ broke) and $3$ ($M$ has it all) are **absorbing**.

**Transition graph.** A birth–death chain: from an interior state $M$ steps **up** by a dollar (winning a
game) with probability $\tfrac23$, or **down** by a dollar with probability $\tfrac13$; the two ends absorb
(the ⟲1 marks a self-loop of probability $1$).
```
    1 ⟲                               1 ⟲
         1/3          1/3
       ◄──────     ◄──────
  [0]*        [1]         [2]         [3]*
                 ──────►      ──────►
                   2/3          2/3
   edges:  [0]* ⟲1              [3]* ⟲1
           [1] ─1/3►[0]   [1] ─2/3►[2]
           [2] ─1/3►[1]   [2] ─2/3►[3]
```
**Transition matrix** (rows/cols $0,1,2,3$):
$$P=\begin{pmatrix}1&0&0&0\\ \tfrac13&0&\tfrac23&0\\ 0&\tfrac13&0&\tfrac23\\ 0&0&0&1\end{pmatrix}.$$

**First-step equations** for $a_m=P(\text{reach }3\mid m)$, with $a_0=0,\ a_3=1$:
$$a_1=\tfrac13 a_0+\tfrac23 a_2=\tfrac23 a_2,\qquad a_2=\tfrac13 a_1+\tfrac23 a_3=\tfrac13 a_1+\tfrac23.$$
Substituting, $a_1=\tfrac23(\tfrac13 a_1+\tfrac23)=\tfrac29 a_1+\tfrac49\Rightarrow\tfrac79 a_1=\tfrac49$, so
$$a_1=\boxed{\tfrac47}\approx0.571,\qquad a_2=\tfrac67.$$
Even starting with only one dollar, $M$'s per-game edge $p=\tfrac23>\tfrac12$ makes him the overall
favorite, since $a_1=\tfrac47>\tfrac12$. More generally, for win probability $p\neq\tfrac12$ with ratio
$r=\tfrac{1-p}{p}$, starting stake $i$, and target $N$, the classic gambler's-ruin formula is
$$a_i=\frac{1-r^{\,i}}{1-r^{\,N}},$$
which here ($r=\tfrac12,\ i=1,\ N=3$) gives $a_1=\dfrac{1-(1/2)^{1}}{1-(1/2)^{3}}=\dfrac{1/2}{7/8}=\tfrac47$.

In [2]:
from fractions import Fraction
import random

def gamblers_ruin_chain(start, total, p_win):
    """Build the gambler's-ruin Markov chain on money in {0,...,total} (0 and `total` absorbing) and return
    P(reach `total` before 0 | start) via the absorption equations."""
    p = Fraction(p_win)
    P = {0: {0: Fraction(1)}, total: {total: Fraction(1)}}
    for m in range(1, total):
        P[m] = {m - 1: 1 - p, m + 1: p}
    a, _ = absorbing_analysis(P, list(range(total + 1)), targets={total})
    return a[start]

def gamblers_ruin_closed(i, N, p):
    """Closed form: fair game -> i/N, else (1 - r^i)/(1 - r^N) with r = (1-p)/p."""
    p = Fraction(p)
    if p == Fraction(1, 2):
        return Fraction(i, N)
    r = (1 - p) / p
    return (1 - r ** i) / (1 - r ** N)

def gamblers_ruin_sim(start=1, total=3, p_win=2 / 3, trials=200000, seed=0):
    rng = random.Random(seed); wins = 0
    for _ in range(trials):
        m = start
        while 0 < m < total:
            m += 1 if rng.random() < p_win else -1
        wins += (m == total)
    return wins / trials

ans = gamblers_ruin_chain(1, 3, Fraction(2, 3))
print("P(M wins | $1, target $3, p=2/3) =", ans, "=", float(ans))
print("  closed form:", gamblers_ruin_closed(1, 3, Fraction(2, 3)), " | simulated:", round(gamblers_ruin_sim(), 4))
for st in (1, 2):
    print(f"  start ${st}: chain {gamblers_ruin_chain(st, 3, Fraction(2,3))}")

P(M wins | $1, target $3, p=2/3) = 4/7 = 0.5714285714285714
  closed form: 4/7  | simulated: 0.5703
  start $1: chain 4/7
  start $2: chain 6/7


### 5.1.2 Dice question

**Problem.** Two dice are rolled repeatedly and the sums recorded. Player $A$ bets a sum of **$12$** appears
first; player $B$ bets **two consecutive $7$s** appear first. What is $P(A\text{ wins})$?

**Per-roll probabilities.** For two dice, $P(\text{sum}=12)=\tfrac1{36}$, $P(\text{sum}=7)=\tfrac6{36}$, and
"anything else" $=\tfrac{29}{36}$.

**States.** $B$ needs *two $7$s in a row*, so the chain must remember whether the **previous roll was a $7$**:
- $S_0$ — last roll was not a $7$ (also the start);
- $S_1$ — last roll **was** a $7$ (one down, one to go);
- $A$ — a $12$ has appeared ($A$ wins), absorbing;
- $B$ — two $7$s in a row ($B$ wins), absorbing.

**Transition graph.**
```
             29/36 (other)
            ┌──────────┐
            ▼          │
   [A]* ◄─1/36─ [S0] ─6/36─► [S1] ─6/36─► [B]*
                  ▲            │  │
                  └────29/36───┘  └─1/36─► [A]*
   edges:  [S0] ─1/36►[A]   [S0] ─6/36►[S1]   [S0] ─29/36►[S0]
           [S1] ─1/36►[A]   [S1] ─6/36►[B]    [S1] ─29/36►[S0]
```
**First-step equations** for $a=P(A\text{ wins})$ from each state ($a_A=1,\ a_B=0$):
$$a_{S_0}=\tfrac1{36}+\tfrac6{36}a_{S_1}+\tfrac{29}{36}a_{S_0},\qquad
a_{S_1}=\tfrac1{36}+\tfrac6{36}\cdot0+\tfrac{29}{36}a_{S_0}.$$
From the first, $7\,a_{S_0}=1+6\,a_{S_1}$; from the second, $36\,a_{S_1}=1+29\,a_{S_0}$. Eliminating
$a_{S_1}$ gives $78\,a_{S_0}=42$, so
$$P(A\text{ wins})=a_{S_0}=\boxed{\tfrac{7}{13}}\approx0.538.$$
Surprisingly $A$ is the favorite: although a $12$ ($\tfrac1{36}$) is far rarer than a $7$ ($\tfrac6{36}$), $B$
must hit the $7$ **twice in a row**, and the frequent "other" rolls keep resetting that streak back to $S_0$.

In [3]:
from fractions import Fraction
import random

def dice_race_prob():
    """P(A wins): a sum of 12 appears before two consecutive 7s. State S0 = last roll not a 7, S1 = last a 7."""
    p12, p7 = Fraction(1, 36), Fraction(6, 36)
    po = 1 - p12 - p7
    P = {'S0': {'A': p12, 'S1': p7, 'S0': po},
         'S1': {'A': p12, 'B': p7, 'S0': po},
         'A': {'A': Fraction(1)}, 'B': {'B': Fraction(1)}}
    a, _ = absorbing_analysis(P, ['S0', 'S1', 'A', 'B'], targets={'A'})
    return a['S0']

def dice_race_sim(trials=300000, seed=0):
    rng = random.Random(seed); Awin = 0
    for _ in range(trials):
        prev7 = False
        while True:
            s = rng.randint(1, 6) + rng.randint(1, 6)
            if s == 12:
                Awin += 1; break
            if s == 7:
                if prev7:
                    break            # two 7s in a row -> B wins
                prev7 = True
            else:
                prev7 = False
    return Awin / trials

ans = dice_race_prob()
print("P(A wins: 12 before two consecutive 7s) =", ans, "=", float(ans))
print("  simulated:", round(dice_race_sim(), 4))

P(A wins: 12 before two consecutive 7s) = 7/13 = 0.5384615384615384
  simulated: 0.5376


### 5.1.3 Coin triplets

Three linked questions about patterns in fair-coin tosses. The unifying trick is a Markov chain whose state is
**how much of the target pattern the recent tosses have already built** — the longest suffix of what we have
tossed that is a prefix of the pattern. From each state a toss either extends the progress, completes the
pattern, or falls back (possibly using an **overlap**).

#### Part A — expected tosses to see a pattern

**Question.** How many tosses on average to first see **HHH**? And to first see **THH**?

**Pattern HHH.** States by progress $\varnothing,\text{H},\text{HH}$ (then HHH ends it). A tail **T** wipes all
progress (T is not a prefix of HHH), so it always resets to $\varnothing$:
```
   [∅] ─H(1/2)─► [H] ─H(1/2)─► [HH] ─H(1/2)─► (HHH) done
    ▲             │             │
    └────T(1/2)───┴────T(1/2)───┘   (any T resets to ∅)
   (∅ also self-loops on T)
```
With $e_s=E[\text{tosses to HHH}\mid s]$:
$$e_{HH}=1+\tfrac12\cdot0+\tfrac12 e_\varnothing,\quad e_{H}=1+\tfrac12 e_{HH}+\tfrac12 e_\varnothing,\quad
e_\varnothing=1+\tfrac12 e_H+\tfrac12 e_\varnothing.$$
Solving gives $e_\varnothing=\boxed{14}$ tosses $(=2+4+8)$.

**Pattern THH.** Now a **T never fully wastes progress**: after reaching TH, a tail returns you to state T (that
T can start a fresh THH), and once in T another T just stays in T:
```
   [∅] ─T(1/2)─► [T] ─H(1/2)─► [TH] ─H(1/2)─► (THH) done
    ▲ ⟲H(1/2)    ▲ ⟲T(1/2)      │
    │            └──────T(1/2)───┘   (TH --T--> T, not ∅)
    └────────────  (∅ --H--> ∅)
```
$$e_{TH}=1+\tfrac12\cdot0+\tfrac12 e_{T},\quad e_{T}=1+\tfrac12 e_{TH}+\tfrac12 e_{T},\quad
e_\varnothing=1+\tfrac12 e_T+\tfrac12 e_\varnothing.$$
Solving gives $e_{T}=6,\ e_{TH}=4$, and $e_\varnothing=\boxed{8}$ tosses.

**Why HHH is slower ($14$ vs $8$).** HHH **overlaps itself**: a run of heads keeps every head useful, but a
single tail throws away *all* the built-up heads at once. THH cannot self-overlap on its head part, so a tail
only ever costs you the last step. (Conway's shortcut: the expected wait is $\sum_k 2^{k}$ over the lengths $k$
where the pattern's prefix equals its suffix — HHH matches at $k=1,2,3\Rightarrow2{+}4{+}8{=}14$; THH only at
$k=3\Rightarrow8$.)

In [4]:
from fractions import Fraction
import random

def expected_wait(pattern, p_head=Fraction(1, 2)):
    """Expected number of tosses to first see `pattern` (a string of 'H'/'T'). State = longest suffix of the
    tosses so far that is a prefix of `pattern`; solved with the expected-time absorption equations."""
    L = len(pattern); pr = {'H': Fraction(p_head), 'T': 1 - Fraction(p_head)}
    prefixes = [pattern[:k] for k in range(L)]                 # '', p0, p0p1, ...
    def nxt(s, c):
        t = s + c
        if t == pattern:
            return pattern
        for k in range(min(len(t), L - 1), -1, -1):           # longest suffix that is a prefix
            if t[len(t) - k:] == pattern[:k]:
                return pattern[:k]
    states = prefixes + [pattern]
    P = {s: {} for s in prefixes}
    for s in prefixes:
        for c in 'HT':
            d = nxt(s, c); P[s][d] = P[s].get(d, 0) + pr[c]
    P[pattern] = {pattern: Fraction(1)}
    _, t = absorbing_analysis(P, states, targets={pattern})
    return t['']

def wait_sim(pattern, trials=200000, seed=0):
    rng = random.Random(seed); tot = 0
    for _ in range(trials):
        seq = ''; n = 0
        while not seq.endswith(pattern):
            seq += 'H' if rng.random() < 0.5 else 'T'; n += 1
        tot += n
    return tot / trials

for pat in ('HHH', 'THH'):
    print(f"E[tosses to {pat}] = {expected_wait(pat)}   (simulated {wait_sim(pat):.3f})")
print("also, e.g. E[HT] =", expected_wait('HT'), ", E[HTH] =", expected_wait('HTH'))

E[tosses to HHH] = 14   (simulated 14.000)
E[tosses to THH] = 8   (simulated 7.983)
also, e.g. E[HT] = 4 , E[HTH] = 10


#### Part B — which comes first, HHH or THH?

**Question.** Keep flipping until **either** HHH or THH appears. What is $P(\text{HHH first})$?

**Combined chain.** Track the progress that is relevant to *both* patterns; the reachable states are
$\varnothing,\text{H},\text{HH},\text{T},\text{TH}$, with absorbing wins **HHH** and **THH**.
```
   [∅] ─H─► [H] ─H─► [HH] ─H─► (HHH win)
    │        │         │
    │T       │T        │T
    ▼        ▼         ▼
   [T] ◄─────┴─────────┘        (any T from the H-side drops to T)
    │⟲T
    │H
    ▼
   [TH] ─H─► (THH win)   ,   [TH] ─T─► [T]
   (every edge has probability 1/2)
```
**First-step equations** for $p_s=P(\text{HHH first}\mid s)$, with $p_{HHH}=1,\ p_{THH}=0$:
$$p_{HH}=\tfrac12\cdot1+\tfrac12 p_T,\quad p_H=\tfrac12 p_{HH}+\tfrac12 p_T,\quad p_\varnothing=\tfrac12 p_H+\tfrac12 p_T,$$
$$p_{TH}=\tfrac12\cdot0+\tfrac12 p_T,\quad p_T=\tfrac12 p_{TH}+\tfrac12 p_T.$$
The last pair forces $p_T=p_{TH}=0$: **once any tail appears, THH is unstoppable before HHH.** Then
$p_{HH}=\tfrac12,\ p_H=\tfrac14$, and
$$P(\text{HHH first})=p_\varnothing=\tfrac12\cdot\tfrac14+\tfrac12\cdot0=\boxed{\tfrac18}.$$
The intuition: to get HHH first you essentially must throw **HHH on the very first three tosses** ($\tfrac18$).
Any earlier tail plants the "T" that THH needs, and THH then completes on the next two heads before a third
head could ever extend to HHH. So THH wins with probability $\tfrac78$.

In [5]:
from fractions import Fraction
import random

def prob_pattern_before(A, B, p_head=Fraction(1, 2)):
    """P(pattern A appears strictly before pattern B) in a fair(ish)-coin stream, via first-step analysis on
    the combined progress states (longest suffix that is a prefix of A or of B)."""
    pr = {'H': Fraction(p_head), 'T': 1 - Fraction(p_head)}
    prefixes = {''}
    for Q in (A, B):
        for k in range(1, len(Q)):
            prefixes.add(Q[:k])
    def nxt(s, c):
        t = s + c
        if t.endswith(A): return 'AW'
        if t.endswith(B): return 'BW'
        for k in range(len(t), -1, -1):
            if t[len(t) - k:] in prefixes:
                return t[len(t) - k:]
    states = list(prefixes) + ['AW', 'BW']
    P = {s: {} for s in prefixes}
    for s in prefixes:
        for c in 'HT':
            d = nxt(s, c); P[s][d] = P[s].get(d, 0) + pr[c]
    P['AW'] = {'AW': Fraction(1)}; P['BW'] = {'BW': Fraction(1)}
    a, _ = absorbing_analysis(P, states, targets={'AW'})
    return a['']

def race_sim(A, B, trials=200000, seed=0):
    rng = random.Random(seed); Aw = 0
    for _ in range(trials):
        seq = ''
        while True:
            seq += 'H' if rng.random() < 0.5 else 'T'
            if seq.endswith(A): Aw += 1; break
            if seq.endswith(B): break
    return Aw / trials

p = prob_pattern_before('HHH', 'THH')
print("P(HHH before THH) =", p, "=", float(p), "  | P(THH first) =", 1 - p)
print("  simulated P(HHH first):", round(race_sim('HHH', 'THH'), 4))

P(HHH before THH) = 1/8 = 0.125   | P(THH first) = 7/8
  simulated P(HHH first): 0.1251


#### Part C — both players choose (Penney's game)

**Question.** Player 1 picks a triplet and **announces** it; player 2 then picks a **different** triplet; whoever's
triplet appears first wins. Both are perfectly rational. What is **player 2's** probability of winning?

**Key fact — the game is non-transitive.** "Appears first" is not a total order: for any triplet player 1 names,
player 2 can name one that beats it. The optimal reply to player 1's $b_1b_2b_3$ is
$$\text{player 2}=(\overline{b_2})\,b_1 b_2\qquad(\overline{b_2}=\text{the opposite of the middle symbol}),$$
which "hooks" onto the front of player 1's pattern. Because player 2 moves second, they always hold the whip
hand — the only question is how badly, and player 1 (rational) picks the triplet that **minimizes** player 2's
edge.

**The full odds table** (player 2's win probability with the optimal reply — the code computes every entry with
`prob_pattern_before`):

| player 1 | player 2 (best reply) | P(player 2 wins) |
|---|---|---|
| HHH | THH | $7/8$ |
| HHT | THH | $3/4$ |
| HTH | HHT | $2/3$ |
| HTT | HHT | $2/3$ |
| THH | TTH | $2/3$ |
| THT | TTH | $2/3$ |
| TTH | HTT | $3/4$ |
| TTT | HTT | $7/8$ |

**Minimax.** A rational player 1 avoids HHH/HHT/TTH/TTT (which hand player 2 $\tfrac34$ or $\tfrac78$) and picks
one of HTH, HTT, THH, THT — the best defense — holding player 2 down to the smallest available edge:
$$P(\text{player 2 wins})=\boxed{\tfrac23}.$$
So even against a perfect opponent, going second is worth $2:1$ odds. (This is why you should always let your
friend call their triplet first.)

In [6]:
from fractions import Fraction
import itertools

pats = [''.join(t) for t in itertools.product('HT', repeat=3)]

def best_reply(p1):
    """Player 2's payoff-maximizing triplet against announced p1, and the resulting P(player 2 wins)."""
    return max(((b, prob_pattern_before(b, p1)) for b in pats if b != p1), key=lambda kv: kv[1])

print("Penney's game -- player 1 announces, player 2 replies optimally:")
rows = []
for p1 in pats:
    b, pr = best_reply(p1)
    rows.append((p1, b, pr))
    print(f"   P1={p1}  ->  P2={b}   P2 wins {pr} = {float(pr):.3f}")

p1_opt, p2_pick, p2_prob = min(rows, key=lambda r: r[2])     # rational P1 minimizes P2's edge
print(f"\nrational P1 plays {p1_opt} (a minimax choice); P2 replies {p2_pick} and wins "
      f"{p2_prob} = {float(p2_prob):.4f}")

Penney's game -- player 1 announces, player 2 replies optimally:
   P1=HHH  ->  P2=THH   P2 wins 7/8 = 0.875
   P1=HHT  ->  P2=THH   P2 wins 3/4 = 0.750
   P1=HTH  ->  P2=HHT   P2 wins 2/3 = 0.667
   P1=HTT  ->  P2=HHT   P2 wins 2/3 = 0.667
   P1=THH  ->  P2=TTH   P2 wins 2/3 = 0.667
   P1=THT  ->  P2=TTH   P2 wins 2/3 = 0.667
   P1=TTH  ->  P2=HTT   P2 wins 3/4 = 0.750
   P1=TTT  ->  P2=HTT   P2 wins 7/8 = 0.875

rational P1 plays HTH (a minimax choice); P2 replies HHT and wins 2/3 = 0.6667


### 5.1.4 Color balls

**Problem.** A box holds $n$ balls, each initially a **different** color. Repeatedly: pick an ordered pair of
balls at random, **repaint the first to match the second**, and return both. What is the expected number of
steps until **all balls share one color**?

**State = the partition of colors.** By symmetry only the multiset of color-group **sizes** matters (a partition
of $n$). Start at $(1,1,\dots,1)$; absorb at $(n)$. Take $n=3$ (groups written largest-first):

- From $(1,1,1)$: whichever ordered pair is chosen, one ball adopts another's color, giving two-of-a-kind — so
  it moves to $(2,1)$ **with probability 1**.
- From $(2,1)$ (colors $X,X,Y$): of the $3\times2=6$ ordered pairs, the two that repaint the lone $Y$ ball to
  $X$ — i.e. (first $=Y$, second $=$ an $X$) — finish the job $\to(3)$; the other four leave a $2\!-\!1$ split.
  So $(2,1)\to(3)$ w.p. $\tfrac26=\tfrac13$ and stays at $(2,1)$ w.p. $\tfrac23$.

**Transition graph ($n=3$).**
```
                       2/3 (self)
                      ┌────────┐
                      ▼        │
   [(1,1,1)] ──1──► [(2,1)] ───┘ ──1/3──► [(3)]*
```
**Expected-time equations** ($t_{(3)}=0$):
$$t_{(2,1)}=1+\tfrac23 t_{(2,1)}\ \Rightarrow\ t_{(2,1)}=3,\qquad
t_{(1,1,1)}=1+t_{(2,1)}=\boxed{4}.$$

**General $n$.** Running the same partition chain for each $n$ (the code builds it and solves
$(I-Q)t=\mathbf 1$ exactly) gives $1,4,9,16,25,\dots$ — that is,
$$E[\text{steps}]=\boxed{(n-1)^{2}}.$$
Intuitively the process is a *voter model* on the complete graph: the count of monochromatic pairs
$\Phi=\sum_c\binom{X_c}{2}$ drifts up by $E[\Delta\Phi]=1-\tfrac{2\Phi}{n(n-1)}$ each step, mean-reverting toward
the absorbed value $\binom n2$; solving the chain turns that drift into the clean $(n-1)^2$.

In [7]:
from fractions import Fraction
import random

def _partitions(n, mx=None):
    """All integer partitions of n as tuples sorted largest-first."""
    if mx is None: mx = n
    if n == 0:
        yield (); return
    for first in range(min(n, mx), 0, -1):
        for rest in _partitions(n - first, first):
            yield (first,) + rest

def color_balls_expected_exact(n):
    """Exact expected steps until one color remains, by building the partition-size Markov chain (pick an
    ordered pair of the n(n-1) ordered pairs; move one ball from its group to the other's) and solving the
    expected-time equations."""
    states = list(_partitions(n)); absorb = (n,); tot = n * (n - 1)
    P = {p: {} for p in states}
    for p in states:
        if p == absorb:
            P[p] = {p: Fraction(1)}; continue
        acc, k = {}, len(p)
        for a in range(k):
            for b in range(k):
                if a == b:
                    w = p[a] * (p[a] - 1)                       # both balls same group -> no change
                    if w: acc[p] = acc.get(p, 0) + w
                else:
                    w = p[a] * p[b]                             # first from group a, second from group b
                    if not w: continue
                    ns = list(p); ns[a] -= 1; ns[b] += 1
                    ns = tuple(sorted((x for x in ns if x > 0), reverse=True))
                    acc[ns] = acc.get(ns, 0) + w
        P[p] = {s: Fraction(v, tot) for s, v in acc.items()}
    _, t = absorbing_analysis(P, states, targets={absorb})
    return t[tuple([1] * n)]

def color_balls_closed(n):
    """Closed form."""
    return (n - 1) ** 2

def color_balls_sim(n, trials=20000, seed=0):
    rng = random.Random(seed); tot = 0
    for _ in range(trials):
        colors = list(range(n)); steps = 0
        while len(set(colors)) > 1:
            i = rng.randrange(n)
            j = rng.randrange(n)
            while j == i:
                j = rng.randrange(n)
            colors[i] = colors[j]; steps += 1        # first ball repainted to second's color
        tot += steps
    return tot / trials

print("n : exact (partition chain) | (n-1)^2 | simulated")
for n in range(2, 7):
    exact = color_balls_expected_exact(n)
    sim = f"{color_balls_sim(n):.2f}" if n <= 5 else "  -"
    print(f"{n} : {str(exact):>4}                      | {color_balls_closed(n):>3}     | {sim}")

n : exact (partition chain) | (n-1)^2 | simulated
2 :    1                      |   1     | 1.00
3 :    4                      |   4     | 3.98
4 :    9                      |   9     | 8.94
5 :   16                      |  16     | 16.00
6 :   25                      |  25     |   -


## 5.2 Martingales and Random Walks

### Random walk
A **random walk** simply adds up independent steps. Starting from $S_0$ (usually $0$),
$$S_t=S_0+X_1+X_2+\cdots+X_t,\qquad X_k\ \text{i.i.d.}$$
- **Simple random walk:** every step is $\pm1$ — up with probability $p$, down with $1-p$. It has a **drift**
  $E[X_k]=p-(1-p)=2p-1$ per step (a biased walk that tends to wander one way).
- **Symmetric random walk:** the fair case $p=\tfrac12$, so $E[X_k]=0$ — **no drift**. (Gambler's ruin in
  §5.1.1 was a simple walk; the drunk man below is symmetric.)

*Example.* Five fair $\pm1$ steps from $0$ might trace $0\to1\to0\to-1\to0\to1$. On average it goes nowhere,
$E[S_t]=0$, yet its spread grows with time: $\operatorname{Var}(S_t)=t$.

### Martingale
A process $M_0,M_1,\dots$ is a **martingale** if, given all history up to now, the expected next value equals
the current one:
$$E[M_{t+1}\mid M_0,\dots,M_t]=M_t.$$
This is the mathematics of a **fair game**: nothing you already know lets you predict a gain or a loss, and
$E[M_t]=E[M_0]$ for every $t$.

**A symmetric random walk is a martingale.** Because the next step has mean $0$ and is independent of the past,
$$E[S_{t+1}\mid\mathcal F_t]=S_t+E[X_{t+1}]=S_t.$$
So a fair $\pm1$ walk is the prototype martingale. (A *biased* walk is **not** a martingale — it drifts; instead
$S_t-(2p-1)t$ is.) Riding on top of the symmetric walk is a second, extremely useful martingale:
$$E[S_{t+1}^2\mid\mathcal F_t]=E[(S_t\pm1)^2\mid\mathcal F_t]=S_t^2+1
\quad\Longrightarrow\quad S_t^2-t\ \text{is a martingale}.$$
These two — $S_t$ and $S_t^2-t$ — crack most symmetric-walk problems: the first delivers exit
**probabilities**, the second delivers expected **time**.

### Stopping rule (stopping time)
A **stopping time** $N$ is a rule for *when to stop* that uses only the information available so far — no
peeking into the future. "Stop the first time the walk hits $+a$ or $-b$" is a stopping time; "stop one step
before the peak" is not. Every problem in this section stops at such a rule (first time a level, a pattern, or
a count is reached).

### Wald's equality
If $N$ is a stopping time for i.i.d. steps $X_k$ (finite mean) with $E[N]<\infty$, the expected total equals
the expected number of steps times the mean step:
$$\boxed{\,E[S_N]=E[N]\,E[X]\,},\qquad S_N=X_1+\cdots+X_N.$$
Intuitively you take $E[N]$ steps on average, each worth $E[X]$; the random *count* and the random *steps*
separate cleanly precisely because a stopping time cannot anticipate future steps.

### The optional-stopping theorem — "a stopped martingale is a martingale"
Stopping a martingale by a stopping time never breaks its fairness: the **stopped process** $M_{t\wedge N}$ is
again a martingale, and under mild conditions ($N$ bounded, or $E[N]<\infty$ with bounded steps),
$$\boxed{\,E[M_N]=E[M_0]\,}.$$
For a symmetric walk this reads $E[S_N]=0$ and $E[S_N^2-N]=0$, i.e. $E[N]=E[S_N^2]$ — the two lines every
problem below reuses.

**Worked solution (fair $\pm1$ walk, stop at $+2$ or $-2$).** Start at $0$ and stop at $N=$ first hit of
$+2$ or $-2$.
- *Fairness of the stopped walk* gives $E[S_N]=0$. Since $S_N\in\{+2,-2\}$ with probabilities $q,1-q$,
  $\;2q-2(1-q)=0\Rightarrow q=\tfrac12$ (equally likely, as symmetry demands).
- *The second martingale* gives $E[S_N^2-N]=0$, so $E[N]=E[S_N^2]=2^2=4$ (matching the general
  $E[N]=\alpha\beta=2\cdot2$).
- *Wald's equality* is consistent: $E[S_N]=E[N]\,E[X]=E[N]\cdot0=0$. ✓

The code cell checks both facts by simulation, and §5.2.1 applies these very two lines to the drunk man.

In [8]:
import random

def rw_exit_sim(a=2, b=2, trials=200000, seed=0):
    """Symmetric +/-1 walk from 0, stopped when it first hits +a or -b.
    Returns (P(hit +a first), E[steps]) -- exact values are b/(a+b) and a*b."""
    rng = random.Random(seed); up = 0; total = 0
    for _ in range(trials):
        s = 0; n = 0
        while -b < s < a:
            s += 1 if rng.random() < 0.5 else -1; n += 1
        up += (s == a); total += n
    return up / trials, total / trials

q, EN = rw_exit_sim(2, 2)
print(f"stop at +2/-2:  P(hit +2) = {q:.3f}  (exact 1/2)   |   E[N] = {EN:.3f}  (exact a*b = 4)")
q, EN = rw_exit_sim(5, 3)
print(f"stop at +5/-3:  P(hit +5) = {q:.3f}  (exact b/(a+b) = 3/8 = 0.375)   |   E[N] = {EN:.2f}  (exact 15)")

stop at +2/-2:  P(hit +2) = 0.500  (exact 1/2)   |   E[N] = 3.996  (exact a*b = 4)
stop at +5/-3:  P(hit +5) = 0.376  (exact b/(a+b) = 3/8 = 0.375)   |   E[N] = 14.94  (exact 15)


### 5.2.1 Drunk man

**Problem.** A drunk man stands at the $17$-metre mark of a $100$-metre bridge and staggers $\pm1$ metre each
step with equal probability. What is the probability he reaches the **far end** (metre $100$) before the
**start** (metre $0$)? And the expected number of steps until he first reaches either end?

**Formulate as a symmetric walk.** Shift the origin to his start: let $Y_t=(\text{position})-17$, a symmetric
$\pm1$ walk from $0$ that stops the first time it hits $+83$ (the far end) or $-17$ (the start). Let $N$ be
that stopping time and $q=P(\text{hit }+83\text{ first})$.
```
   metre 0                      metre 17                      metre 100
   [start]* ◄─────────────────── ● (drunk) ───────────────► [end]*
    Y = -17            each step ±1, prob 1/2               Y = +83
   (both ends are absorbing)
```
**Exit probability — first martingale $S_t$.** The stopped walk is a martingale, so $E[Y_N]=Y_0=0$. With
$Y_N\in\{+83,-17\}$,
$$83\,q-17\,(1-q)=0\ \Longrightarrow\ 100\,q=17\ \Longrightarrow\ q=\boxed{0.17}.$$

**Expected time — second martingale $S_t^2-t$.** Since $Y_t^2-t$ is a martingale, $E[Y_N^2-N]=0$, hence
$$E[N]=E[Y_N^2]=q\cdot83^2+(1-q)\cdot17^2=0.17\cdot6889+0.83\cdot289=\boxed{1411}.$$
(General symmetric walk from $0$ stopping at $+\alpha$ or $-\beta$: $q=\dfrac{\beta}{\alpha+\beta}$ and
$E[N]=\alpha\beta$; here $\dfrac{17}{100}=0.17$ and $83\cdot17=1411$.)

**Note on the book's number.** The Green Book prints $E[N]=1441$; the correct value is
$\alpha\beta=83\cdot17=1411$ (an arithmetic slip — $1441$ transposes the digits), confirmed by the simulation
below.

In [9]:
from fractions import Fraction
import random

def drunk_man(start=17, length=100):
    """Symmetric walk on [0, length] from `start`. Returns (P(reach far end first), E[steps]).
    alpha = length - start (distance to far end), beta = start (distance to near end)."""
    alpha, beta = length - start, start
    q = Fraction(beta, alpha + beta)          # P(hit +alpha before -beta) = beta/(alpha+beta)
    EN = alpha * beta                         # expected steps for a symmetric walk
    return q, EN

def drunk_sim(start=17, length=100, trials=40000, seed=0):
    rng = random.Random(seed); reach = 0; total = 0
    for _ in range(trials):
        x = start; n = 0
        while 0 < x < length:
            x += 1 if rng.random() < 0.5 else -1; n += 1
        reach += (x == length); total += n
    return reach / trials, total / trials

q, EN = drunk_man(17, 100)
ps, es = drunk_sim(17, 100)
print(f"P(reach metre 100 first) = {q} = {float(q)}   (simulated {ps:.3f})")
print(f"E[steps to reach either end] = {EN}   (simulated {es:.0f});  book prints 1441, correct is 83*17 = 1411")

P(reach metre 100 first) = 17/100 = 0.17   (simulated 0.170)
E[steps to reach either end] = 1411   (simulated 1400);  book prints 1441, correct is 83*17 = 1411


### 5.2.2 Dice game (via Wald's equality)

**Problem.** Roll a die and collect its face value; on a $4,5,6$ you may roll again, on a $1,2,3$ the game
stops. What is the expected total payoff? (Solved with the tower rule in §4.5.3; here is the one-line
martingale/Wald argument the book intends in Chapter 5.)

**Formulate with a stopping time.** Each roll independently **continues** with probability $\tfrac12$ (a
$4,5,6$) and **stops** with probability $\tfrac12$ (a $1,2,3$), so the number of rolls $N$ is
**geometric** with $p=\tfrac12$ — a clean stopping rule with
$$E[N]=\frac1p=2.$$
The payoff of a single roll has mean $E[X]=\dfrac{1+2+3+4+5+6}{6}=\dfrac72$. The rolls are i.i.d. and $N$ is a
stopping time, so **Wald's equality** applies directly:
$$E[S_N]=E[N]\,E[X]=2\cdot\tfrac72=\boxed{7}.$$
This is exactly the answer from §4.5.3, obtained here without setting up any recursion — the whole content is
"expected number of rolls $\times$ expected value per roll".

In [10]:
from fractions import Fraction
import random

def dice_game_wald(faces=tuple(range(1, 7)), continue_faces=(4, 5, 6)):
    """Expected payoff via Wald: N ~ Geometric(p_stop) so E[N] = 1/p_stop; E[payoff] = E[N]*E[face]."""
    p_stop = Fraction(len(faces) - len(continue_faces), len(faces))
    E_N = 1 / p_stop
    E_X = Fraction(sum(faces), len(faces))
    return E_N, E_X, E_N * E_X

def dice_game_sim(trials=400000, seed=0):
    rng = random.Random(seed); total = 0
    for _ in range(trials):
        pay = 0
        while True:
            r = rng.randint(1, 6); pay += r
            if r <= 3:
                break
        total += pay
    return total / trials

E_N, E_X, payoff = dice_game_wald()
print(f"E[N] = {E_N} rolls,  E[X] = {E_X} per roll  ->  E[payoff] = E[N]*E[X] = {payoff} = {float(payoff)}")
print(f"  simulated: {dice_game_sim():.4f}")

E[N] = 2 rolls,  E[X] = 7/2 per roll  ->  E[payoff] = E[N]*E[X] = 7 = 7.0
  simulated: 6.9861


### 5.2.3 Ticket line (reflection principle)

**Problem.** $2n$ people queue for $5$-dollar tickets: $n$ carry a $5$-dollar bill and $n$ carry a $10$-dollar
bill, and the seller starts with **no change**. Each buys one ticket. What is the probability the whole line is
served without anyone ever having to wait for change (the seller always has a $5$ to hand back to a
$10$-payer)?

**Formulate as a lattice walk.** Give each $5$-dollar payer a step $+1$ (a $5$ goes into the till) and each
$10$-dollar payer a step $-1$ (a $5$ must come out as change). The seller can always make change **iff every
partial sum stays $\ge0$**. So we count lattice paths from $(0,0)$ to $(2n,0)$ ($n$ up-steps, $n$ down-steps)
that **never dip below $0$**. All arrangements: $\binom{2n}{n}$.
```
  step  +1 = a 5-dollar payer (till gains a 5)    -1 = a 10-dollar payer (needs a 5)
  OK  iff  the running total never goes below 0:

    1 |    __        __
    0 |___/  \__/\__/  \___   (stays >= 0  =>  change always available)
   -1 |........forbidden.........
```
**Reflection principle.** Count the **bad** paths — those that touch $y=-1$. For each, reflect the part of the
path *after its first visit to $-1$* across the line $y=-1$. This is a bijection between bad paths and
*unrestricted* paths that end at $-2$ (i.e. $n-1$ up-steps and $n+1$ down-steps), of which there are
$\binom{2n}{n-1}$. Therefore
$$\#\text{good}=\binom{2n}{n}-\binom{2n}{n-1},\qquad
P=\frac{\binom{2n}{n}-\binom{2n}{n-1}}{\binom{2n}{n}}=1-\frac{n}{n+1}=\boxed{\frac{1}{n+1}}.$$
The count of good paths is exactly the **Catalan number** $C_n=\dfrac1{n+1}\dbinom{2n}{n}$.

In [11]:
from fractions import Fraction
from math import comb
import random

def ticket_line_prob(n):
    """P(all 2n people served without waiting for change), via reflection:
    good = C(2n,n) - C(2n,n-1);  P = good / C(2n,n) = 1/(n+1)  (Catalan)."""
    total = comb(2 * n, n)
    good = comb(2 * n, n) - comb(2 * n, n - 1)
    return Fraction(good, total)

def ticket_line_sim(n, trials=200000, seed=0):
    rng = random.Random(seed); ok = 0
    for _ in range(trials):
        seq = [1] * n + [-1] * n; rng.shuffle(seq)
        s = 0; good = True
        for v in seq:
            s += v
            if s < 0:
                good = False; break
        ok += good
    return ok / trials

for n in (1, 2, 3, 5):
    p = ticket_line_prob(n)
    extra = f"   (simulated {ticket_line_sim(n):.4f})" if n <= 3 else ""
    print(f"n={n}: reflection gives {p} = {float(p):.4f}   [= 1/(n+1) = {Fraction(1, n+1)}]{extra}")

n=1: reflection gives 1/2 = 0.5000   [= 1/(n+1) = 1/2]   (simulated 0.4997)
n=2: reflection gives 1/3 = 0.3333   [= 1/(n+1) = 1/3]   (simulated 0.3327)
n=3: reflection gives 1/4 = 0.2500   [= 1/(n+1) = 1/4]   (simulated 0.2512)
n=5: reflection gives 1/6 = 0.1667   [= 1/(n+1) = 1/6]


### 5.2.4 Coin sequence

**Problem.** For a fair coin, what is the expected number of tosses to first get **$n$ heads in a row**?

**Approach 1 — induction (as in §4.4 for the HHH / THH triplets).** Let $E_n$ be the answer. To get $n$ heads
in a row you must first reach $n-1$ in a row ($E_{n-1}$ tosses), then toss once more: with probability
$\tfrac12$ it is H and you are done, with probability $\tfrac12$ it is T and you start over. Hence
$$E_n=E_{n-1}+1+\tfrac12E_n\ \Longrightarrow\ E_n=2E_{n-1}+2,\quad E_0=0,$$
so $E_1=2,\ E_2=6,\ E_3=14,\dots$ and in closed form
$$E_n=2^{\,n+1}-2=2+4+\cdots+2^{\,n}.$$

**Approach 2 — the martingale (casino) argument with a stream of gamblers.** Picture a fair casino. Just
before each toss, **a new gambler enters** and bets one dollar that this toss is H; the game pays fair $2\!:\!1$, so a
correct call turns a stake $s$ into $2s$ and a wrong call loses it. A winning gambler **lets it all ride**,
betting the entire pot that the run of heads continues, and is wiped out the moment a tail breaks the run.
Because every individual bet is fair, each gambler's fortune is a martingale — and so is the **total** fortune
(what the casino owes minus what it has taken in), which starts at $0$.

**The stopping rule and the stopped martingale.** Stop the whole game at $N=$ the first time some gambler has
ridden a full run of $n$ heads (equivalently, the first time $n$ consecutive heads appear). $N$ is a stopping
time, so by optional stopping the total fortune stays a mean-zero martingale:
$$\underbrace{E[\text{money paid in}]}_{\text{one 1-dollar stake per toss}}
=\underbrace{E[\text{money paid out}]}_{\text{fortunes of gamblers still alive at }N}.$$
Money paid in after $N$ tosses is exactly $N$ dollars (one gambler per toss). At the instant the run of $n$
heads completes, the gamblers still alive are precisely those who entered $1,2,\dots,n$ tosses ago — the one
who entered $k$ tosses back has watched $k$ heads and holds $2^{k}$. So the money paid out is
$2^{1}+2^{2}+\cdots+2^{n}=2^{\,n+1}-2$, and fairness gives
$$E[N]=2^{\,n+1}-2,$$
the same answer, now with the mechanism exposed: **each way the pattern overlaps itself keeps one gambler
alive, contributing its $2^{k}$.**

In [12]:
from fractions import Fraction
import random

def nheads_induction(n):
    """E[tosses to n heads in a row] by the recursion E_n = 2 E_{n-1} + 2, E_0 = 0  (= 2^{n+1} - 2)."""
    E = 0
    for _ in range(n):
        E = 2 * E + 2
    return E

def nheads_martingale(n):
    """Same expectation by the stopped-martingale payout: gamblers alive at the stop hold 2^1+...+2^n."""
    return sum(2 ** k for k in range(1, n + 1))

def nheads_sim(n, trials=200000, seed=0):
    rng = random.Random(seed); tot = 0
    for _ in range(trials):
        run = 0; t = 0
        while run < n:
            t += 1
            run = run + 1 if rng.random() < 0.5 else 0
        tot += t
    return tot / trials

print(" n | induction | martingale (2^1+..+2^n) | closed 2^(n+1)-2 | Markov expected_wait | sim")
for n in (1, 2, 3, 4, 5):
    ind, mar = nheads_induction(n), nheads_martingale(n)
    mk = expected_wait("H" * n)                       # the KMP-Markov solver from 5.1.3, as a cross-check
    sim = f"{nheads_sim(n):.2f}" if n <= 4 else "  -"
    print(f" {n} | {ind:>9} | {mar:>22} | {2**(n+1)-2:>16} | {str(mk):>20} | {sim}")

 n | induction | martingale (2^1+..+2^n) | closed 2^(n+1)-2 | Markov expected_wait | sim
 1 |         2 |                      2 |                2 |                    2 | 2.00
 2 |         6 |                      6 |                6 |                    6 | 5.99
 3 |        14 |                     14 |               14 |                   14 | 14.00
 4 |        30 |                     30 |               30 |                   30 | 30.02
 5 |        62 |                     62 |               62 |                   62 |   -


#### The same stopped martingale for a general pattern — HHTTHH

The gambler stream does not care that the target was all heads. For **any** pattern $P=b_1b_2\cdots b_L$, each
gambler bets in turn on $b_1,b_2,\dots$; at the stopping time (first appearance of $P$) the gamblers still
alive are exactly those who entered $k$ tosses ago **such that the last $k$ tosses equal both a suffix of $P$
and its own prefix $b_1\cdots b_k$** — that is, the lengths $k$ at which $P$'s prefix equals its suffix (its
self-overlaps). Each such gambler holds $2^{k}$, so reading off the payoff when the sequence stops,
$$E[N]=\sum_{k:\ \text{prefix}_k=\text{suffix}_k}2^{k}.$$

For $P=\mathtt{HHTTHH}$ (length $6$) the prefix equals the suffix at $k=6$ (the whole word $\mathtt{HHTTHH}$),
at $k=2$ ($\mathtt{HH}$), and at $k=1$ ($\mathtt{H}$) — and at no other $k$. Reading the payoff after the
sequence stops,
$$E[N]=2^{6}+2^{2}+2^{1}=64+4+2=\boxed{70}\ \text{tosses}.$$
The all-heads pattern is just the special case where **every** $k$ overlaps, recovering
$2+4+\cdots+2^{n}=2^{\,n+1}-2$.

In [13]:
from fractions import Fraction
import random

def stopped_martingale_wait(pattern):
    """E[tosses to first see `pattern`] by the stopped-martingale / self-overlap sum:
    sum of 2^k over lengths k where pattern's prefix of length k equals its suffix of length k."""
    L = len(pattern)
    overlaps = [k for k in range(1, L + 1) if pattern[:k] == pattern[L - k:]]
    return sum(2 ** k for k in overlaps), overlaps

def wait_sim(pattern, trials=300000, seed=0):
    rng = random.Random(seed); tot = 0
    for _ in range(trials):
        seq = ''; n = 0
        while not seq.endswith(pattern):
            seq += 'H' if rng.random() < 0.5 else 'T'; n += 1
        tot += n
    return tot / trials

for pat in ('HHTTHH', 'HHH', 'THH', 'HTH'):
    val, ov = stopped_martingale_wait(pat)
    payoff = " + ".join(f"2^{k}" for k in reversed(ov))
    mk = expected_wait(pat)                            # cross-check against the 5.1.3 Markov solver
    sim = f"{wait_sim(pat):.2f}" if pat in ('HHTTHH', 'HHH') else " -"
    print(f"{pat:>7}: overlaps k={ov}  ->  {payoff} = {val}   (Markov {mk}, sim {sim})")

 HHTTHH: overlaps k=[1, 2, 6]  ->  2^6 + 2^2 + 2^1 = 70   (Markov 70, sim 70.07)
    HHH: overlaps k=[1, 2, 3]  ->  2^3 + 2^2 + 2^1 = 14   (Markov 14, sim 14.02)
    THH: overlaps k=[3]  ->  2^3 = 8   (Markov 8, sim  -)
    HTH: overlaps k=[1, 3]  ->  2^3 + 2^1 = 10   (Markov 10, sim  -)


## 5.3 Dynamic Programming

**Dynamic programming (DP)** cracks multi-stage decision problems by splitting them into a chain of one-step
choices and solving them **backwards**, from the last stage to the first. It is the engine behind the four
games here — and, as the final part shows, behind option pricing.

### The underlying discrete-time system
A DP problem plays out over **stages** $k=0,1,\dots,N-1$. At each stage:
- the situation is summarised by a **state** $x_k$ (everything the future needs to know — the same
  "memoryless" idea as §5.1);
- you pick a **decision / control** $u_k$ from the allowed choices;
- a random **disturbance** $w_k$ is revealed (the roll, the card, the game result);
- the state advances by a fixed **transition rule**
$$\boxed{\,x_{k+1}=f_k(x_k,u_k,w_k)\,}.$$

*Simple example — the 3-roll dice game (§5.3.1).* Stage $k$ = which roll you are on; state $x_k$ = the face
currently in hand; decision $u_k\in\{\text{keep},\ \text{roll again}\}$; disturbance $w_k$ = the next face
$1,\dots,6$. Transition: *keep* ends the game holding $x_k$; *roll again* sets $x_{k+1}=w_{k+1}$, a fresh face
(the old one forgone).

### The cost / profit function (additive over stages)
DP needs the objective to be a **sum across stages** of per-stage rewards plus a terminal reward:
$$\text{total}=\sum_{k=0}^{N-1} g_k(x_k,u_k,w_k)\;+\;g_N(x_N).$$
Additivity is the key structure: the total always splits into "reward now" $+$ "reward from here on", which is
exactly what the recursion exploits. (In the dice games each stage's reward is $0$ until you stop or bust, and
the terminal reward is what you bank; in option pricing each stage just discounts the next.)

### The principle of optimality
> **Principle of optimality (Bellman).** Whatever the first decision and the state it leads to, the
> *remaining* decisions must themselves be optimal for the sub-problem starting from that new state.

In one line: **the tail of an optimal plan is itself optimal.** So we never search whole strategies at once —
solve the last stage, then the second-to-last assuming optimal play afterwards, and so on backwards.

### The basic DP algorithm (backward induction)
Let $J_k(x)$ be the best expected total reward obtainable from stage $k$ onward in state $x$ — the **value
function** (or "cost-to-go"). Start at the end and recurse **backwards**:
$$J_N(x)=g_N(x),\qquad
\boxed{\,J_k(x)=\max_{u}\ E_{w}\!\big[\,g_k(x,u,w)+J_{k+1}\big(f_k(x,u,w)\big)\,\big]\,}$$
(use $\min$ for costs). The maximiser $u^{*}(x)$ at each state is the **optimal policy**, and $J_0(x_0)$ is the
value of the whole game. Every problem below is one instance of this single boxed recursion — only the state
and the meaning of "keep vs. continue" change.

*Worked example — the 3-roll dice game.* The only reward is the banked face, so $J$ is the expected banked
amount.
- Last roll ($k=3$, must accept): $J_3=E[\text{face}]=3.5$.
- Second roll: keep the face if it beats the continuation value $3.5$ (keep $4,5,6$):
  $\;J_2=\tfrac16(4+5+6)+\tfrac36(3.5)=4.25$.
- First roll: keep if the face beats $4.25$ (keep $5,6$):
  $\;J_1=\tfrac16(5+6)+\tfrac46(4.25)=\tfrac{14}{3}\approx4.67$.

So the game is worth $J_1=\tfrac{14}{3}$, with policy "keep $5,6$ after roll 1; keep $4,5,6$ after roll 2." The
next cell does this for any number of rolls and any die.

### 5.3.1 Dice game

**Problem.** Roll a die up to **3 times**. After the first or second roll you may keep the current face $x$
(take $x$ dollars) or roll again — but rolling again forfeits the face you just saw. On the third roll you must
take whatever comes up. What is the game worth, and what is the strategy?

**DP formulation.** This is the concept worked example above. With $f(r)$ the value when $r$ rolls remain,
$$f(1)=\tfrac{s+1}{2}\ (\text{must accept}),\qquad f(r)=\tfrac1s\sum_{\text{face}=1}^{s}\max\big(\text{face},\,f(r-1)\big).$$
For a standard die ($s=6$): $f(1)=3.5,\ f(2)=4.25,\ f(3)=\tfrac{14}{3}$. **Value $=\tfrac{14}{3}\approx4.67$**;
keep a face only when it beats the continuation value (keep $5,6$ after roll 1, keep $4,5,6$ after roll 2).

In [14]:
from fractions import Fraction

def dice_game(rolls=3, sides=6):
    """Value of the 'roll up to `rolls` times, keep-or-continue' game via backward induction.
    Returns (value, thresholds) where thresholds[k] is the continuation value you compare the face against
    on the k-th-from-first decision (keep the face iff face > threshold)."""
    v = Fraction(sides + 1, 2)                      # f(1): last roll, must accept -> mean face
    thresholds = []
    for _ in range(rolls - 1):
        thresholds.append(v)                        # keep current face iff face > v
        v = sum(max(Fraction(face), v) for face in range(1, sides + 1)) / sides
    thresholds.reverse()
    return v, thresholds

val, thr = dice_game(3, 6)
print(f"3 rolls, d6: value = {val} = {float(val):.4f}")
print("  keep-if-face-exceeds thresholds (roll 1, roll 2):", [str(t) for t in thr])
for rolls in (1, 2, 3, 4, 5):
    v, _ = dice_game(rolls, 6)
    print(f"  up to {rolls} rolls: value = {v} = {float(v):.4f}")

3 rolls, d6: value = 14/3 = 4.6667
  keep-if-face-exceeds thresholds (roll 1, roll 2): ['17/4', '7/2']
  up to 1 rolls: value = 7/2 = 3.5000
  up to 2 rolls: value = 17/4 = 4.2500
  up to 3 rolls: value = 14/3 = 4.6667
  up to 4 rolls: value = 89/18 = 4.9444
  up to 5 rolls: value = 277/54 = 5.1296


### 5.3.2 World series

**Problem.** A best-of-7 series (first to $4$ wins). You have $100$ to make, and may only bet
**double-or-nothing on each individual game** (stake $y$: gain $+y$ if the Red Sox win that game, lose $y$ if
they lose). Choose per-game stakes so you finish at exactly $+100$ if the Red Sox win the series and $-100$ if
they lose. How much do you bet on each game?

**DP formulation.** State $(i,j)=$ (Red Sox wins, Rockies wins). Let $f(i,j)$ be the net payoff the strategy
must be worth there. The goal pins the terminals:
$$f(4,j)=+100\ (j<4),\qquad f(i,4)=-100\ (i<4).$$
Betting $y$ on the next game, a Sox win must land you at $f(i+1,j)$ and a loss at $f(i,j+1)$:
$$f(i,j)+y=f(i+1,j),\quad f(i,j)-y=f(i,j+1)
\ \Longrightarrow\ f(i,j)=\frac{f(i+1,j)+f(i,j+1)}{2},\quad y(i,j)=\frac{f(i+1,j)-f(i,j+1)}{2}.$$
The value equation is a **pure expectation** (average of the two successors, *no* max) — fair games, so
$f(0,0)=0$: the hedge is free. The stake $y(i,j)$ is the **spread** between the two successor payoffs — this
is the *delta* of dynamic hedging, and it returns in §5.3.5. Backward induction gives the first bet
$$y(0,0)=\frac{f(1,0)-f(0,1)}{2}=\boxed{31.25}.$$

In [15]:
from fractions import Fraction

def world_series(win=4, stake=100):
    """Double-or-nothing hedge over a best-of-(2*win-1) series. Returns (f, bets) over states (i,j):
    f(i,j) = required net payoff = (f(i+1,j)+f(i,j+1))/2 ; bet y(i,j) = (f(i+1,j)-f(i,j+1))/2."""
    f = {}
    for j in range(win): f[(win, j)] = Fraction(stake)
    for i in range(win): f[(i, win)] = Fraction(-stake)
    def val(i, j):
        if (i, j) not in f:
            f[(i, j)] = (val(i + 1, j) + val(i, j + 1)) / 2
        return f[(i, j)]
    val(0, 0)
    bets = {(i, j): (val(i + 1, j) - val(i, j + 1)) / 2 for i in range(win) for j in range(win)}
    return f, bets

f, bets = world_series()
print("f(0,0) =", f[(0, 0)], " (a fair, cost-free hedge)")
print("first bet y(0,0) =", bets[(0, 0)], "=", float(bets[(0, 0)]))
print("\nbet table y(i,j)  (rows i = Sox wins, cols j = Rockies wins):")
print("      j=0     j=1     j=2     j=3")
for i in range(4):
    print(f"  i={i}: " + "  ".join(f"{float(bets[(i,j)]):6.2f}" for j in range(4)))

f(0,0) = 0  (a fair, cost-free hedge)
first bet y(0,0) = 125/4 = 31.25

bet table y(i,j)  (rows i = Sox wins, cols j = Rockies wins):
      j=0     j=1     j=2     j=3
  i=0:  31.25   31.25   25.00   12.50
  i=1:  31.25   37.50   37.50   25.00
  i=2:  25.00   37.50   50.00   50.00
  i=3:  12.50   25.00   50.00  100.00


### 5.3.3 Dynamic dice game

**Problem.** Roll a die repeatedly. A $1$ pays $1$, a $2$ pays $2$, …, a $5$ pays $5$ (added to your running
total), but a $6$ **wipes out everything** and ends the game. After any non-$6$ roll you may **stop and keep
your total** or roll on. Risk-neutral, what is the game worth?

**DP formulation.** State $n=$ money accumulated; decision $=$ stop or roll. Rolling once more from $n$ has
expected value $\tfrac16(n{+}1)+\cdots+\tfrac16(n{+}5)+\tfrac16\cdot0=\tfrac56 n+2.5$, after which you again
play optimally, so
$$f(n)=\max\Big(n,\ \tfrac16\textstyle\sum_{i=1}^{5} f(n+i)\Big).$$
**Where to stop.** Rolling beats stopping only while $\tfrac56 n+2.5>n$, i.e. $n<15$. So **stop once
$n\ge15$**; the most you can bank is $19$ (a $5$ rolled at $n=14$). Hence $f(n)=n$ for $15\le n\le19$, and for
$n\le14$, $f(n)=\tfrac16\sum_{i=1}^5 f(n+i)$ (the $\tfrac16\cdot0$ for a six drops out). Solving backwards,
$$f(0)=\boxed{6.15},$$
so a risk-neutral player pays at most about $6.15$. (The code generalises: for an $s$-sided die, stop once
$n\ge s(s-1)/2$.)

In [16]:
from fractions import Fraction

def dynamic_dice(sides=6):
    """Value f(0) of the accumulate-or-bust dice game (top face busts). Stop once n >= sides*(sides-1)/2.
    Returns (f0, stop_threshold, max_payoff, f) with f exact (Fraction)."""
    stop = sides * (sides - 1) // 2
    top = stop + sides - 2                          # largest bankable total
    f = {n: Fraction(n) for n in range(stop, top + 1)}
    for n in range(stop - 1, -1, -1):
        f[n] = sum(f[n + i] for i in range(1, sides)) / sides   # face `sides` -> 0
    return f[0], stop, top, f

f0, stop, top, f = dynamic_dice(6)
print(f"stop once n >= {stop};  max bankable = {top};  value f(0) = {float(f0):.4f}  (pay at most this)")
print("  f(n) for n = 19..10:", [round(float(f[n]), 2) for n in range(19, 9, -1)])
print("  8-sided die: f(0) =", round(float(dynamic_dice(8)[0]), 4))

stop once n >= 15;  max bankable = 19;  value f(0) = 6.1537  (pay at most this)
  f(n) for n = 19..10: [19.0, 18.0, 17.0, 16.0, 15.0, 14.17, 13.36, 12.59, 11.85, 11.16]
  8-sided die: f(0) = 11.1814


### 5.3.4 Dynamic card game

**Problem.** A shuffled standard deck ($26$ red, $26$ black) is drawn one card at a time, **without**
replacement. Each **red** pays $+1$, each **black** pays $-1$; you may stop any time. What is the optimal
stopping rule, and the value of the game?

**DP formulation.** State $(b,r)=$ black and red cards **still in the deck**. Since $26-r$ reds and $26-b$
blacks have been drawn, the money banked so far is $(26-r)-(26-b)=b-r$, so **stopping locks in $b-r$**. Drawing
one more card is black w.p. $\tfrac{b}{b+r}$ (to $(b-1,r)$) or red w.p. $\tfrac{r}{b+r}$ (to $(b,r-1)$), giving
the **optimal-stopping** equation
$$f(b,r)=\max\Big(\underbrace{b-r}_{\text{stop}},\ \underbrace{\tfrac{b}{b+r}f(b-1,r)+\tfrac{r}{b+r}f(b,r-1)}_{\text{keep drawing}}\Big),$$
with $f(b,0)=b$ (only blacks left — stop) and $f(0,r)=0$ (only reds left — draw them all back up to $0$). Keep
drawing exactly while the continuation value exceeds $b-r$. Backward induction gives
$$f(26,26)=\boxed{2.62}.$$
That "**max(stop-now, keep-going)**" shape is precisely the early-exercise decision of an **American option**,
developed next.

In [17]:
from functools import lru_cache

def dynamic_card(black=26, red=26):
    """Value of the red/black stopping game with `black` black and `red` red cards left.
    f(b,r) = max( b - r,  (b/(b+r)) f(b-1,r) + (r/(b+r)) f(b,r-1) )."""
    @lru_cache(maxsize=None)
    def f(b, r):
        if b == 0:
            return 0.0                              # only reds left: draw all -> net climbs back to 0
        if r == 0:
            return float(b)                         # only blacks left: stop
        cont = b / (b + r) * f(b - 1, r) + r / (b + r) * f(b, r - 1)
        return max(b - r, cont)                     # stop value b-r vs. keep drawing
    return f(black, red)

def should_stop(b, r):
    """Optimal rule: stop iff banking b-r now is at least the value of drawing on."""
    from functools import lru_cache
    @lru_cache(maxsize=None)
    def f(b, r):
        if b == 0: return 0.0
        if r == 0: return float(b)
        return max(b - r, b/(b+r)*f(b-1, r) + r/(b+r)*f(b, r-1))
    cont = 0.0 if (b == 0 or r == 0) else b/(b+r)*f(b-1, r) + r/(b+r)*f(b, r-1)
    return (b - r) >= cont

print("value f(26,26) =", round(dynamic_card(), 4), " -> pay at most this")
print("  a few states: f(26,26)=%.3f  f(10,5)=%.3f  f(5,10)=%.3f" %
      (dynamic_card(26, 26), dynamic_card(10, 5), dynamic_card(5, 10)))
print("  stop when ahead by enough, e.g. stop at (b=26,r=20)?", should_stop(26, 20),
      "| keep at (b=26,r=25)?", not should_stop(26, 25))

value f(26,26) = 2.6245  -> pay at most this
  a few states: f(26,26)=2.624  f(10,5)=5.000  f(5,10)=0.515
  stop when ahead by enough, e.g. stop at (b=26,r=20)? True | keep at (b=26,r=25)? True


### 5.3.5 American and European options by dynamic programming

The last two problems are two faces of one DP, and both are exactly how options are priced on a **binomial
tree**. Model the stock in small steps: each step it goes **up** to $Su$ or **down** to $Sd$
($u=e^{\sigma\sqrt{\Delta t}},\ d=1/u$), and under the **risk-neutral** measure the up-probability is
$p=\dfrac{e^{r\Delta t}-d}{u-d}$. Then work backward from the payoff at expiry.

**European option $=$ the world-series recursion (pure expectation $+$ a hedge/delta).** A European option is
exercised only at expiry, so an interior node is just the discounted expected value — **no max**:
$$f(\text{node})=e^{-r\Delta t}\big(p\,f_{\text{up}}+(1-p)\,f_{\text{down}}\big).$$
This is the world-series equation $f=\tfrac12(f_{\uparrow}+f_{\downarrow})$ with risk-neutral weights and
discounting. The replicating **hedge** — how much stock to hold — is the delta
$$\Delta=\frac{f_{\text{up}}-f_{\text{down}}}{S_{\text{up}}-S_{\text{down}}},$$
the exact analogue of the world-series bet $y=\tfrac{f(i+1,j)-f(i,j+1)}{2}$.

**American option $=$ the card-game recursion (optimal stopping, *with* a max).** An American option may be
exercised **early**, so at each node you take the better of exercising now or holding — the very same
stop-vs-continue max as the dynamic card game:
$$f(\text{node})=\max\Big(\underbrace{\text{intrinsic value now}}_{\text{exercise}},\
\underbrace{e^{-r\Delta t}\big(p\,f_{\text{up}}+(1-p)\,f_{\text{down}}\big)}_{\text{hold}}\Big).$$

The code prices both by backward induction. The European values match the Black–Scholes formula; the American
**put** is worth **more** than the European put (early exercise has value), while the American **call** on a
non-dividend stock **equals** the European call (early exercise is never optimal).

In [18]:
import math

def norm_cdf(x):
    return 0.5 * (1 + math.erf(x / math.sqrt(2)))

def black_scholes(S, K, r, sigma, T, call=True):
    d1 = (math.log(S / K) + (r + sigma ** 2 / 2) * T) / (sigma * math.sqrt(T))
    d2 = d1 - sigma * math.sqrt(T)
    if call:
        return S * norm_cdf(d1) - K * math.exp(-r * T) * norm_cdf(d2)
    return K * math.exp(-r * T) * norm_cdf(-d2) - S * norm_cdf(-d1)

def binomial_option(S, K, r, sigma, T, steps=500, call=True, american=False):
    """CRR binomial option price by backward induction.
    European: node = disc*(p*up + (1-p)*down)  (the world-series pure-expectation recursion).
    American: node = max(intrinsic, hold)       (the card-game stop-vs-continue recursion)."""
    dt = T / steps
    u = math.exp(sigma * math.sqrt(dt)); d = 1 / u
    p = (math.exp(r * dt) - d) / (u - d); disc = math.exp(-r * dt)
    intr = lambda s: max((s - K) if call else (K - s), 0.0)
    val = [intr(S * u ** j * d ** (steps - j)) for j in range(steps + 1)]     # payoff at expiry
    for i in range(steps - 1, -1, -1):
        val = [disc * (p * val[j + 1] + (1 - p) * val[j]) for j in range(i + 1)]
        if american:
            val = [max(val[j], intr(S * u ** j * d ** (i - j))) for j in range(i + 1)]
    return val[0]

S, K, r, sigma, T = 100, 100, 0.05, 0.20, 1.0
print(f"S={S} K={K} r={r} sigma={sigma} T={T}, binomial with 500 steps:")
print(f"  European call: {binomial_option(S,K,r,sigma,T,500,True ,False):7.4f}   (Black-Scholes {black_scholes(S,K,r,sigma,T,True):.4f})")
print(f"  European put : {binomial_option(S,K,r,sigma,T,500,False,False):7.4f}   (Black-Scholes {black_scholes(S,K,r,sigma,T,False):.4f})")
print(f"  American call: {binomial_option(S,K,r,sigma,T,500,True ,True ):7.4f}   (= European call: no early exercise on a non-dividend call)")
print(f"  American put : {binomial_option(S,K,r,sigma,T,500,False,True ):7.4f}   (> European put: early exercise is worth something)")

S=100 K=100 r=0.05 sigma=0.2 T=1.0, binomial with 500 steps:
  European call: 10.4466   (Black-Scholes 10.4506)
  European put :  5.5695   (Black-Scholes 5.5735)
  American call: 10.4466   (= European call: no early exercise on a non-dividend call)
  American put :  6.0888   (> European put: early exercise is worth something)


## 5.4 Brownian Motion and Stochastic Calculus

The continuous-time limit of the random walk is **Brownian motion**, and the calculus that goes with it —
**Itô's lemma** — is the language of modern finance. This section builds the three ideas in turn (Brownian
motion, first-passage/stopping times, Itô's lemma) and answers the standard interview questions under each.

### 5.4.1 Brownian motion — definition and properties (1A)

A **standard Brownian motion** (or **Wiener process**) $B_t$ is the canonical *continuous-time, continuous-path*
stochastic process. It is defined by four properties:
1. **$B_0=0$.**
2. **Independent increments:** for $s<t$, the increment $B_t-B_s$ is independent of the whole past
   $\{B_u:u\le s\}$.
3. **Stationary Gaussian increments:** $B_t-B_s\sim N(0,\,t-s)$ — mean $0$, variance equal to the time
   elapsed.
4. **Continuous sample paths** (though nowhere differentiable — the paths are infinitely jagged).

From these follow the moments $E[B_t]=0$, $\operatorname{Var}(B_t)=t$, and the covariance
$\operatorname{Cov}(B_s,B_t)=\min(s,t)$.

**Two structural properties matter most.**
- **Martingale (the key one).** Because the next increment has mean $0$ and is independent of the past,
$$E[B_t\mid\mathcal F_s]=B_s+E[B_t-B_s\mid\mathcal F_s]=B_s+0=B_s\qquad(s<t),$$
so $B_t$ is a **martingale** — a fair game, exactly as in §5.2. This is what lets optional stopping crack the
hitting-time problems below, and it is the property option-pricing leans on (discounted prices are
martingales under the risk-neutral measure).
- **Markov.** Since increments are independent, the future $\{B_u:u>t\}$ depends on the past only through the
present value $B_t$ — the memoryless property of §5.1, now in continuous time.

Other standard facts: $B_t$ is **Gaussian** (any finite collection $(B_{t_1},\dots,B_{t_n})$ is jointly
normal), **self-similar** ($c^{-1/2}B_{ct}$ is again a Brownian motion), and has **quadratic variation**
$[B]_t=t$ — informally $(dB_t)^2=dt$, the single rule that drives all of Itô calculus.

### 5.4.2 Correlation of a Brownian motion and its square (1B)

**Question.** What is the correlation between $B_t$ and $B_t^{2}$?

**Step 1 — the covariance.** $\operatorname{Cov}(B_t,B_t^2)=E[B_t\cdot B_t^2]-E[B_t]E[B_t^2]=E[B_t^3]-0$.
Since $B_t\sim N(0,t)$ is symmetric about $0$, **every odd moment vanishes**, so $E[B_t^3]=0$ and
$$\operatorname{Cov}(B_t,B_t^2)=0.$$

**Step 2 — the correlation.** A zero covariance forces a zero correlation (the variances are finite and
positive: $\operatorname{Var}(B_t)=t$ and $\operatorname{Var}(B_t^2)=E[B_t^4]-(E[B_t^2])^2=3t^2-t^2=2t^2$):
$$\rho(B_t,B_t^2)=\frac{0}{\sqrt{t\cdot 2t^2}}=\boxed{0}.$$
So $B_t$ and $B_t^2$ are **uncorrelated** for every $t$ — even though they are obviously *dependent*
($B_t^2$ is a function of $B_t$). It is the classic reminder that *uncorrelated $\ne$ independent*: symmetry
kills the linear relationship while the nonlinear one remains.

In [19]:
import random, math

def bsq_correlation_sim(t=1.0, n=1000000, seed=0):
    """Monte-Carlo corr(B_t, B_t^2). Exact value is 0 because Cov = E[B^3] = 0 (odd moment)."""
    rng = random.Random(seed)
    b = [rng.gauss(0, math.sqrt(t)) for _ in range(n)]
    b2 = [x * x for x in b]
    mb, mb2 = sum(b) / n, sum(b2) / n
    cov = sum((b[i] - mb) * (b2[i] - mb2) for i in range(n)) / n
    vb = sum((x - mb) ** 2 for x in b) / n
    vb2 = sum((x - mb2) ** 2 for x in b2) / n
    return cov / math.sqrt(vb * vb2)

print("corr(B_t, B_t^2) Monte-Carlo:", round(bsq_correlation_sim(), 4), "  (exact 0)")
print("check: Var(B^2) = 3t^2 - t^2 = 2t^2, and Cov = E[B^3] = 0")

corr(B_t, B_t^2) Monte-Carlo: 0.002   (exact 0)
check: Var(B^2) = 3t^2 - t^2 = 2t^2, and Cov = E[B^3] = 0


### 5.4.3 Probability that $B_1>0$ and $B_2<0$ (1C)

**Question.** For a Brownian motion, find $P(B_1>0\ \text{and}\ B_2<0)$.

**Solution 1 — the bivariate-normal integral.** $(B_1,B_2)$ is jointly Gaussian with mean $0$ and covariance
matrix $\begin{pmatrix}\operatorname{Var}(B_1)&\operatorname{Cov}(B_1,B_2)\\ \operatorname{Cov}(B_1,B_2)&\operatorname{Var}(B_2)\end{pmatrix}=\begin{pmatrix}1&1\\1&2\end{pmatrix}$
(using $\operatorname{Cov}(B_1,B_2)=\min(1,2)=1$), so their correlation is $\rho=\dfrac{1}{\sqrt{1\cdot2}}=\dfrac{1}{\sqrt2}$.
For a standard bivariate normal, $P(X>0,Y<0)=\dfrac14-\dfrac{\arcsin\rho}{2\pi}$. With
$\arcsin\tfrac1{\sqrt2}=\tfrac\pi4$,
$$P(B_1>0,B_2<0)=\frac14-\frac{\pi/4}{2\pi}=\frac14-\frac18=\boxed{\frac18}.$$

**Solution 2 — independence + a symmetry picture (cleaner).** Split $B_2=B_1+(B_2-B_1)$. By property 2,
$U:=B_1\sim N(0,1)$ and $V:=B_2-B_1\sim N(0,1)$ are **independent**. The event becomes
$$\{B_1>0,\ B_2<0\}=\{U>0,\ U+V<0\}.$$
Because $U$ and $V$ are independent standard normals, their joint density is **rotationally symmetric** (it
depends only on $u^2+v^2$). The region $\{u>0,\ u+v<0\}$ is the wedge between the ray $u=0$ (pointing down)
and the ray $u+v=0$ (pointing down-right) — an angle of exactly $45^\circ$. A $45^\circ$ slice of a
rotationally symmetric density carries probability $\dfrac{45^\circ}{360^\circ}=\boxed{\dfrac18}$.

<p align="center">
<svg viewBox="0 0 340 320" width="360" xmlns="http://www.w3.org/2000/svg" style="max-width:100%;height:auto">
  <circle cx="150" cy="140" r="110" fill="none" stroke="currentColor" stroke-opacity="0.25"/>
  <path d="M150 140 L227.8 217.8 A110 110 0 0 1 150.0 250.0 Z" fill="#e69138" fill-opacity="0.55" stroke="#b45f06"/>
  <line x1="26" y1="140" x2="274" y2="140" stroke="currentColor" stroke-width="1.2"/>
  <line x1="150" y1="16" x2="150" y2="268" stroke="currentColor" stroke-width="1.2"/>
  <line x1="72.2" y1="62.2" x2="227.8" y2="217.8" stroke="#3d6ea5" stroke-dasharray="4 3" stroke-opacity="0.9"/>
  <text x="278" y="144" font-size="12" fill="currentColor" font-style="italic">B&#8321;</text>
  <text x="156" y="24" font-size="12" fill="currentColor" font-style="italic">B&#8322;&#8722;B&#8321;</text>
  <text x="232" y="232" font-size="11" fill="#3d6ea5">B&#8322;=0</text>
  <text x="184" y="220" font-size="11" fill="#b45f06">45&#176; wedge</text>
  <text x="12" y="308" font-size="12" fill="currentColor">both axes are independent N(0,1) &#8658; density is rotationally symmetric, so P(B&#8321;&gt;0, B&#8322;&lt;0) = 45&#176;/360&#176; = 1/8</text>
</svg>
</p>

Both the $B_1$ axis and the $B_2-B_1$ axis carry a standard-normal density; it is precisely their
independence (so the picture is circularly symmetric) that turns the answer into a matter of measuring the
angle.

In [20]:
import random, math

def prob_pos_neg_sim(n=1000000, seed=0):
    """P(B1 > 0 and B2 < 0), using B2 = B1 + (B2 - B1) with the two parts iid N(0,1)."""
    rng = random.Random(seed); c = 0
    for _ in range(n):
        u = rng.gauss(0, 1)          # B1
        v = rng.gauss(0, 1)          # B2 - B1  (independent of B1)
        if u > 0 and u + v < 0:      # B1 > 0 and B2 < 0
            c += 1
    return c / n

print("P(B1>0, B2<0) Monte-Carlo:", round(prob_pos_neg_sim(), 4), "  (exact 1/8 = 0.125)")
print("correlation of (B1,B2) = 1/sqrt(2); arcsin = pi/4  ->  1/4 - 1/8 = 1/8")

P(B1>0, B2<0) Monte-Carlo: 0.1246   (exact 1/8 = 0.125)
correlation of (B1,B2) = 1/sqrt(2); arcsin = pi/4  ->  1/4 - 1/8 = 1/8


### 5.4.4 Mean stopping time to $\pm1$ (2A)

**First passage & stopping times.** A **stopping time** $\tau$ for Brownian motion is (as in §5.2) a rule to
stop that peeks no further than the present; the **first-passage time** to a level or set is the prototype.
The two martingales $B_t$ and $B_t^2-t$ turn these into one-line calculations by optional stopping.

**Question.** Let $\tau$ be the first time $B_t$ hits $-1$ or $+1$. Find $E[\tau]$.

**Step 1 — show $B_t^2-t$ is a martingale.** By Itô's lemma on $f(B)=B^2$ (using $(dB)^2=dt$),
$$d(B_t^2)=2B_t\,dB_t+\tfrac12\cdot2\,(dB_t)^2=2B_t\,dB_t+dt\quad\Longrightarrow\quad B_t^2-t=\int_0^t 2B_s\,dB_s,$$
an Itô integral, hence a **martingale** (mean-zero, no drift). (Directly:
$E[B_t^2\mid\mathcal F_s]=B_s^2+(t-s)$, so $E[B_t^2-t\mid\mathcal F_s]=B_s^2-s$.)

**Step 2 — optional stopping.** Applying it at $\tau$, $E[B_\tau^2-\tau]=0$, so $E[\tau]=E[B_\tau^2]$. At
$\tau$ the walk sits at $\pm1$, so $B_\tau^2=1$ with certainty, giving
$$E[\tau]=\boxed{1}.$$
(General symmetric barriers $-a,\,+b$: the same argument gives $E[\tau]=ab$ — the continuous cousin of the
symmetric-walk result in §5.2.)

In [21]:
import random, math

def mean_hit_time_sim(a=1.0, b=1.0, dt=0.001, trials=6000, seed=0):
    """E[first time B_t hits +a or -b]. Exact value is a*b (here 1). (Coarse Euler simulation.)"""
    rng = random.Random(seed); sd = math.sqrt(dt); total = 0.0
    for _ in range(trials):
        x = 0.0; t = 0.0
        while -b < x < a:
            x += rng.gauss(0, sd); t += dt
        total += t
    return total / trials

print("E[tau to +/-1] Monte-Carlo:", round(mean_hit_time_sim(), 3), "  (exact a*b = 1)")

E[tau to +/-1] Monte-Carlo: 1.054   (exact a*b = 1)


### 5.4.5 First-passage density of a Wiener process (2B)

**What is a Wiener process?** "Wiener process" is simply the mathematician's name for **standard Brownian
motion** $W_t$ — the process of §5.4.1 (start at $0$, independent $N(0,t{-}s)$ increments, continuous paths).
Below, $\tau_x=\min\{t: W_t=x\}$ is the first-passage time to level $x>0$.

**Question.** Find the distribution (density) of $\tau_x$ and its expected value.

**Step 1 — reflection principle.** Split $\{W_t\ge x\}$ by whether the path has already reached $x$:
$$P(W_t\ge x)=P(\tau_x\le t,\ W_t\ge x)+P(\tau_x\le t,\ W_t<x).$$
The first term is just $P(\tau_x\le t,\ W_t\ge x)$. For the second, **reflect** the path after $\tau_x$ about
the level $x$: by symmetry a path that reaches $x$ and ends below is equally likely to end above, so
$P(\tau_x\le t,\ W_t<x)=P(\tau_x\le t,\ W_t\ge x)$. Hence
$$P(\tau_x\le t)=2\,P(W_t\ge x)=2\Big(1-\Phi\big(\tfrac{x}{\sqrt t}\big)\Big)=2\,\Phi\!\Big(\!-\tfrac{x}{\sqrt t}\Big).$$

**Step 2 — differentiate for the density.** With $\Phi'=\varphi$ and $\frac{d}{dt}\big(\!-\tfrac{x}{\sqrt t}\big)=\tfrac{x}{2}t^{-3/2}$,
$$f_{\tau_x}(t)=\frac{d}{dt}\,2\Phi\!\Big(\!-\tfrac{x}{\sqrt t}\Big)=\frac{x}{\sqrt{2\pi}\,t^{3/2}}\,e^{-x^{2}/(2t)},\qquad t>0.$$

**Step 3 — the mean.** The tail decays like $t^{-3/2}$, so $t\,f_{\tau_x}(t)\sim t^{-1/2}$, whose integral
diverges:
$$E[\tau_x]=\boxed{\infty}.$$
The level is reached **with probability $1$** ($P(\tau_x\le t)\to1$ as $t\to\infty$), yet the *expected* time
to reach it is infinite — a hallmark of the recurrent-but-null Brownian motion.

In [22]:
import random, math

def first_passage_cdf(x, t):
    """P(tau_x <= t) = 2 * Phi(-x / sqrt(t)) by the reflection principle."""
    return 2 * (0.5 * (1 + math.erf((-x / math.sqrt(t)) / math.sqrt(2))))

def first_passage_pdf(x, t):
    """Density f(t) = x / (sqrt(2 pi) t^{3/2}) exp(-x^2 / (2t))."""
    return x / (math.sqrt(2 * math.pi) * t ** 1.5) * math.exp(-x * x / (2 * t))

def first_passage_sim(x=1.0, T=1.0, dt=0.001, trials=6000, seed=0):
    rng = random.Random(seed); sd = math.sqrt(dt); hit = 0
    for _ in range(trials):
        w = 0.0; t = 0.0
        while t < T:
            w += rng.gauss(0, sd); t += dt
            if w >= x:
                hit += 1; break
    return hit / trials

print(f"P(tau_1 <= 1): exact 2*Phi(-1) = {first_passage_cdf(1,1):.4f}   Monte-Carlo {first_passage_sim():.4f}")
print(f"density at (x=1, t=1): f = {first_passage_pdf(1,1):.4f};   E[tau_x] = infinity (t^-3/2 tail)")

P(tau_1 <= 1): exact 2*Phi(-1) = 0.3173   Monte-Carlo 0.3017
density at (x=1, t=1): f = 0.2420;   E[tau_x] = infinity (t^-3/2 tail)


### 5.4.6 Hitting $3$ before $-5$, with and without drift (2C)

**Question.** $X$ is a driftless Brownian motion, $dX=dW$, $X_0=0$. What is $P(X\text{ hits }3\text{ before }-5)$?
And if $X$ has drift $m$, $dX=m\,dt+dW$?

**No drift.** $X_t$ is a martingale, so by optional stopping $E[X_\tau]=X_0=0$. Writing $p=P(\text{hit }+3\text{ first})$
and using $X_\tau\in\{3,-5\}$,
$$3p-5(1-p)=0\ \Longrightarrow\ 8p=5\ \Longrightarrow\ p=\boxed{\tfrac58}.$$
(General barriers $+a,\,-b$: $p=\dfrac{b}{a+b}$ — the same $b/(a+b)$ as the drunk man.)

**With drift — Feynman–Kac.** Let $u(x)=P_x(\text{hit }3\text{ before }-5)$. Feynman–Kac says $u$ is
**harmonic for the generator** of $X$, i.e. it solves the ODE $\mathcal L u=0$ where
$\mathcal L=m\,\dfrac{d}{dx}+\tfrac12\dfrac{d^2}{dx^2}$, with boundary values $u(3)=1,\ u(-5)=0$:
$$\tfrac12 u''(x)+m\,u'(x)=0.$$
Solving, $u''=-2m\,u'\Rightarrow u'(x)=Ce^{-2mx}\Rightarrow u(x)=A+D\,e^{-2mx}$. Imposing $u(3)=1,\ u(-5)=0$
and evaluating at the start $x=0$,
$$p=u(0)=\frac{e^{2m\cdot5}-1}{e^{2m\cdot5}-e^{-2m\cdot3}}=\boxed{\frac{e^{10m}-1}{e^{10m}-e^{-6m}}}.$$
As $m\to0$ this $\to\dfrac{10}{16}=\dfrac58$, recovering the driftless answer; positive drift pushes $p$ toward
$1$, negative drift toward $0$.

In [23]:
import math, random

def hit_upper_first(a=3.0, b=5.0, m=0.0):
    """P(drifted BM dX = m dt + dW from 0 hits +a before -b).
    m=0: b/(a+b);  else (e^{2mb} - 1)/(e^{2mb} - e^{-2ma})  (Feynman-Kac / scale function)."""
    if abs(m) < 1e-12:
        return b / (a + b)
    return (math.exp(2 * m * b) - 1) / (math.exp(2 * m * b) - math.exp(-2 * m * a))

def hit_sim(a=3.0, b=5.0, m=0.0, dt=0.005, trials=6000, seed=0):
    rng = random.Random(seed); sd = math.sqrt(dt); up = 0
    for _ in range(trials):
        x = 0.0
        while -b < x < a:
            x += m * dt + rng.gauss(0, sd)
        up += (x >= a)
    return up / trials

print(f"no drift: P(hit 3 before -5) = 5/8 = {hit_upper_first(3,5,0):.4f}   (MC {hit_sim(3,5,0.0):.4f})")
for m in (0.2, -0.2):
    print(f"drift m={m:+}: P = {hit_upper_first(3,5,m):.4f}   (MC {hit_sim(3,5,m):.4f})")

no drift: P(hit 3 before -5) = 5/8 = 0.6250   (MC 0.6250)
drift m=+0.2: P = 0.9014   (MC 0.9040)
drift m=-0.2: P = 0.2715   (MC 0.2665)


### 5.4.7 Probability $X$ never reaches $-1$ (2D)

**Question.** $X$ is a generalized Wiener process $dX=dt+dW$ (drift $m=1$), $X_0=0$. What is
$P(X\text{ never reaches }-1)$?

**Solution — send the upper barrier to infinity in 2C.** "Never reaching $-1$" means hitting $+\infty$ before
$-1$, i.e. taking $a\to\infty$ with lower barrier $b=1$ in the drift formula. With $m=1>0$, $e^{-2ma}\to0$, so
$$P(\text{hit }+a\text{ before }-1)=\frac{e^{2m\cdot1}-1}{e^{2m\cdot1}-e^{-2m a}}\ \xrightarrow{a\to\infty}\ \frac{e^{2m}-1}{e^{2m}}=1-e^{-2m}.$$
With $m=1$,
$$P(X\text{ never reaches }-1)=1-e^{-2}\approx\boxed{0.865}.$$
Equivalently, $P(\text{ever hit }-1)=e^{-2m b}=e^{-2}\approx0.135$: a positive drift makes downward excursions
exponentially unlikely, so the walk usually escapes upward forever.

In [24]:
import math, random

def prob_never_reaches(down=1.0, m=1.0):
    """P(dX = m dt + dW from 0 never hits -down) = 1 - e^{-2 m down}  (m > 0)."""
    return 1 - math.exp(-2 * m * down)

def never_sim(down=1.0, m=1.0, dt=0.005, cap=30.0, trials=6000, seed=0):
    rng = random.Random(seed); sd = math.sqrt(dt); never = 0
    for _ in range(trials):
        x = 0.0
        while -down < x < cap:                 # cap stands in for +infinity
            x += m * dt + rng.gauss(0, sd)
        never += (x >= cap)
    return never / trials

print(f"P(never reaches -1) = 1 - e^-2 = {prob_never_reaches():.4f}   (MC, cap=30: {never_sim():.4f})")
print(f"so P(ever hits -1) = e^-2 = {math.exp(-2):.4f}")

P(never reaches -1) = 1 - e^-2 = 0.8647   (MC, cap=30: 0.8738)
so P(ever hits -1) = e^-2 = 0.1353


### 5.4.8 Itô's lemma and the martingale criterion (3A)

**Itô's lemma** is the chain rule of stochastic calculus. If $X$ is an Itô process
$dX=\beta(t,X)\,dt+\gamma(t,X)\,dW$, then for a $C^{2}$ function $f(t,X)$,
$$df=\Big(\underbrace{\frac{\partial f}{\partial t}+\beta\frac{\partial f}{\partial x}+\tfrac12\gamma^{2}\frac{\partial^{2}f}{\partial x^{2}}}_{\text{drift}}\Big)dt+\gamma\frac{\partial f}{\partial x}\,dW.$$
The extra $\tfrac12\gamma^2 f_{xx}$ term — absent from ordinary calculus — comes from $(dW)^2=dt$.

**The martingale criterion.** An Itô process $dX=a(t,x)\,dt+b(t,x)\,dW$ is a **martingale iff its drift
vanishes**, $a(t,x)=0$ (a driftless Itô integral has mean $0$ and no predictable trend). So to test whether
some $f$ is a martingale, apply Itô and read off the $dt$ coefficient.

**Worked example.** Let $Z_t=\sqrt{t}\,B_t$. First its law: $Z_t$ is $\sqrt t$ times $B_t\sim N(0,t)$, so
$$E[Z_t]=0,\qquad \operatorname{Var}(Z_t)=t\cdot\operatorname{Var}(B_t)=t\cdot t=t^2,\qquad Z_t\sim N(0,t^2).$$
Is it a martingale? Apply Itô to $f(t,B)=\sqrt t\,B$ (here $f_t=\tfrac{B}{2\sqrt t}$, $f_B=\sqrt t$,
$f_{BB}=0$):
$$dZ_t=\frac{B_t}{2\sqrt t}\,dt+\sqrt t\,dB_t.$$
The drift $\tfrac{B_t}{2\sqrt t}$ is nonzero whenever $B_t\ne0$ (probability $1$), so **$Z_t=\sqrt t\,B_t$ is
not a martingale** — despite having constant mean $0$. (Constant mean is necessary but *not* sufficient; the
drift must vanish path-by-path.)

In [25]:
import random, math

def Z_moments_sim(t=3.0, n=400000, seed=0):
    """Z_t = sqrt(t) * B_t.  Exact: mean 0, variance t^2 (so Z_t ~ N(0, t^2))."""
    rng = random.Random(seed)
    zs = [math.sqrt(t) * rng.gauss(0, math.sqrt(t)) for _ in range(n)]
    mean = sum(zs) / n
    var = sum((z - mean) ** 2 for z in zs) / n
    return mean, var

m, v = Z_moments_sim(3.0)
print(f"Z_3 = sqrt(3) B_3:  mean {m:+.3f} (exact 0),  variance {v:.2f} (exact t^2 = 9)")
print("Ito drift of Z_t is B_t/(2 sqrt t) != 0  ->  NOT a martingale")

Z_3 = sqrt(3) B_3:  mean +0.010 (exact 0),  variance 9.02 (exact t^2 = 9)
Ito drift of Z_t is B_t/(2 sqrt t) != 0  ->  NOT a martingale


### 5.4.9 Is $W(t)^3$ a martingale? (3B)

**Question.** For a Brownian motion $W_t$, is $W_t^{3}$ a martingale?

**Step 1 — apply Itô** to $f(W)=W^{3}$ (here $f_W=3W^2$, $f_{WW}=6W$, and $\beta=0,\ \gamma=1$):
$$d(W_t^{3})=3W_t^{2}\,dW_t+\tfrac12\cdot6W_t\,dt=3W_t^{2}\,dW_t+3W_t\,dt.$$

**Step 2 — read the drift.** The $dt$ coefficient is $3W_t$, which is **not identically zero**. By the
criterion of §5.4.8, $W_t^{3}$ is therefore **not a martingale**.

**Confirmation by direct conditioning.** For $s<t$, writing $\Delta=W_t-W_s\sim N(0,t-s)$ independent of
$\mathcal F_s$,
$$E[W_t^{3}\mid\mathcal F_s]=E[(W_s+\Delta)^3\mid\mathcal F_s]=W_s^3+3W_s^2\underbrace{E[\Delta]}_{0}+3W_s\underbrace{E[\Delta^2]}_{t-s}+\underbrace{E[\Delta^3]}_{0}=W_s^{3}+3W_s(t-s),$$
which exceeds $W_s^{3}$ whenever $W_s>0$ — not equal, so **not a martingale**. (The fix, if you want one: the
Itô drift shows $W_t^3-3\!\int_0^t W_s\,ds$ *is* a martingale, and more neatly $W_t^3-3tW_t$ is too.)

In [26]:
import random, math

def cond_expectation_W3(w=1.0, s=1.0, t=2.0, band=0.05, trials=4000000, seed=0):
    """Estimate E[W_t^3 | W_s ~ w] by simulation; exact value is w^3 + 3w(t-s), NOT w^3."""
    rng = random.Random(seed); num = 0.0; cnt = 0
    sds = math.sqrt(s); sdd = math.sqrt(t - s)
    for _ in range(trials):
        ws = rng.gauss(0, sds)
        if abs(ws - w) < band:
            wt = ws + rng.gauss(0, sdd)
            num += wt ** 3; cnt += 1
    return num / cnt, cnt

est, cnt = cond_expectation_W3()
print(f"E[W_2^3 | W_1≈1]  ~ {est:.2f}   (exact w^3 + 3w(t-s) = 1 + 3 = 4, not w^3 = 1)")
print(f"drift 3W dt != 0  ->  W^3 is NOT a martingale   (sampled {cnt} paths near W_1=1)")

E[W_2^3 | W_1≈1]  ~ 4.00   (exact w^3 + 3w(t-s) = 1 + 3 = 4, not w^3 = 1)
drift 3W dt != 0  ->  W^3 is NOT a martingale   (sampled 96354 paths near W_1=1)


---
*More of Chapter 5 as I keep reading.*